<div dir="rtl">

# الدرس 44: التضمينات: الكلمات كمتجهات

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### جدول البحث، بيدنا وبباي تورش

</div>

In [ ]:
import numpy as np
import torch
from torch import nn

words = ["قط", "كلب", "سيارة"]
E = np.array([[0.6, 0.8], [0.8, 0.6], [-0.6, 0.8]])

one_hot = np.array([0, 1, 0])
print("one-hot @ E:", one_hot @ E, "  row 1:", E[1])

def cosine(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

for i in range(3):
    for j in range(i + 1, 3):
        print(words[i], words[j], round(cosine(E[i], E[j]), 2))

In [ ]:
emb = nn.Embedding(num_embeddings=3, embedding_dim=2)
with torch.no_grad():
    emb.weight.copy_(torch.tensor(E, dtype=torch.float32))

ids = torch.tensor([1, 0, 2, 1])                  # a "sentence" of token ids
print(emb(ids))
print("requires grad:", emb.weight.requires_grad)
print("GPT-2 embedding parameters:", 50257 * 768)

<div dir="rtl">

### تضمينات حقيقية من نص شكسبير

</div>

In [ ]:
import os
import re
import urllib.request
from collections import Counter
from sklearn.decomposition import TruncatedSVD

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
tokens = re.findall(r"[a-z']+", open("tinyshakespeare.txt", encoding="utf-8").read().lower())

V = 3000                                              # keep the 3000 most frequent words
vocab = [w for w, _ in Counter(tokens).most_common(V)]
index = {w: i for i, w in enumerate(vocab)}
ids = np.array([index.get(w, -1) for w in tokens])     # -1: rare word, ignored

# 1. co-occurrence counts: neighbours up to 2 words away
C = np.zeros((V, V))
for offset in (1, 2):
    a, b = ids[:-offset], ids[offset:]
    keep = (a >= 0) & (b >= 0)
    np.add.at(C, (a[keep], b[keep]), 1)
    np.add.at(C, (b[keep], a[keep]), 1)

# 2. positive PMI: how much more often together than chance
total = C.sum()
expected = C.sum(axis=1, keepdims=True) * C.sum(axis=0, keepdims=True) / total
with np.errstate(divide="ignore", invalid="ignore"):
    ppmi = np.nan_to_num(np.maximum(np.log(C / expected), 0))

# 3. compress 3000 columns into 64 dimensions, then normalize each vector
vectors = TruncatedSVD(n_components=64, random_state=0).fit_transform(ppmi)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
print("embedding matrix:", vectors.shape)

<div dir="rtl">

### أقرب الجيران

</div>

In [ ]:
def neighbours(word, k=6):
    sims = vectors @ vectors[index[word]]
    return [vocab[i] for i in np.argsort(-sims)[1:k + 1]]     # skip the word itself

for w in ["king", "father", "three", "eyes", "night", "sword"]:
    print(f"{w:7} -> {neighbours(w)}")

<div dir="rtl">

### رسم الكلمات

</div>

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
groups = {
    "family": ["father", "mother", "son", "daughter", "brother", "sister", "wife", "husband"],
    "numbers": ["two", "three", "four", "five", "six", "ten", "hundred", "thousand"],
    "body": ["eyes", "hand", "heart", "head", "tongue", "face", "blood", "tears"],
    "time": ["night", "day", "morning", "hour", "year", "days", "hours", "years"],
}
chosen = [w for ws in groups.values() for w in ws]
xy = PCA(n_components=2).fit_transform(vectors[[index[w] for w in chosen]])
plt.figure(figsize=(7.5, 5))
k = 0
for g, (name, ws) in enumerate(groups.items()):
    pts = xy[k:k + len(ws)]; k += len(ws)
    plt.scatter(pts[:, 0], pts[:, 1], s=30, label=name, color=f"C{g}")
    for (x, y), w in zip(pts, ws):
        plt.annotate(w, (x, y), textcoords="offset points", xytext=(4, 3), fontsize=9)
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.show()

<div dir="rtl">

### ماذا عن الحساب بالكلمات؟

</div>

In [ ]:
def analogy(a, b, c, k=4):
    """a is to b as c is to ?"""
    v = vectors[index[b]] - vectors[index[a]] + vectors[index[c]]
    sims = vectors @ (v / np.linalg.norm(v))
    return [vocab[i] for i in np.argsort(-sims) if vocab[i] not in (a, b, c)][:k]

print("man : king  ::  woman : ?", analogy("man", "king", "woman"))
print("he  : his   ::  she   : ?", analogy("he", "his", "she"))

<div dir="rtl">

## تمرين: جيب التمام، ثم جيرانك

1. احسب بيدك تشابه جيب التمام بين المتجهات: $a = (1, 2, 2)$، و $b = (2, 1, 2)$، و $c = (2, -2, 1)$.
2. اختر ثلاث كلمات إنجليزية تظنها شائعة عند شكسبير، واطبع جيرانها. هل يبدو الجيران منطقيين؟

</div>

In [ ]:
# اكتب حلّك هنا
